# benchmarking_main.ipynb
**Scalable Benchmarking Framework for Cell Type Annotation Tools (Python Version)**

This is the main execution notebook that orchestrates the benchmarking process:
1. Loads configuration and tool registry
2. Sources helper functions from benchmarking_helpers.ipynb
3. Provides main execution function run_all_tools_cv()
4. Contains example usage patterns

Supports both marker-based and reference-based methods using k-fold cross-validation with comprehensive result aggregation.

This is a Python/scanpy translation of the R Benchmarking.R file.

## Load Required Libraries

In [1]:
# Load required libraries
import scanpy as sc
import pandas as pd
import numpy as np
import anndata as ad
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import time
import warnings
from typing import Dict, List, Tuple, Any, Optional, Callable
import os
import pickle
from datetime import datetime

## Load Helper Functions

Note: In a Jupyter environment, you would either:
1. Run the benchmarking_helpers.ipynb notebook first, or
2. Import the functions if converted to a .py module

For now, we'll assume the helper functions are available.

In [2]:
# Load helper functions
%run benchmarking_helpers.ipynb  # Uncomment if running in Jupyter

# Alternative: import from .py version
# from benchmarking_helpers import (
#     calculate_metrics, create_cv_folds, get_fold_data_cached,
#     run_single_tool_cv, aggregate_tool_results, aggregate_all_tools
# )

# Import DL-based tool functions
import sys
sys.path.append('./DL-based')

try:
    from run_scDeepSort import run_scDeepSort_function
    from run_scHash import run_scHash_function
    from run_scDeepinsight import run_scDeepinsight_function
    from run_TOSICA import run_TOSICA_function
    from run_scMMT import run_scMMT_function
    from run_scnym import run_scnym_function
    from run_CIForm import run_CIForm_function
    from run_mtANN import run_mtANN_function
    from run_scTransSort import run_scTransSort_function
    from run_scBalance import run_scBalance_function
    from run_MARS import run_MARS_function
    from run_Cell_BLAST import run_Cell_BLAST_function
    from run_SCANVI import run_SCANVI_function
    from run_ItClust import run_ItClust_function
    from run_ACTINN import run_ACTINN_function
    
    print("✓ All Python DL-based tools imported successfully")
except ImportError as e:
    print(f"⚠ Could not import some Python DL tools: {e}")
    # Set None for any missing tools to prevent errors
    run_scDeepSort_function = None
    run_scHash_function = None
    run_scDeepinsight_function = None
    run_TOSICA_function = None
    run_scMMT_function = None
    run_scnym_function = None
    run_CIForm_function = None
    run_mtANN_function = None
    run_scTransSort_function = None
    run_scBalance_function = None
    run_MARS_function = None
    run_Cell_BLAST_function = None
    run_SCANVI_function = None
    run_ItClust_function = None
    run_ACTINN_function = None

#Not working: scDeepInsight (too complex), MARS (too old), scTransSort(broken), mtANN (broken/too long), ACTINN(outdated tensorflow), ItClust(too old)

Benchmarking helper functions loaded successfully!
✓ All Python DL-based tools imported successfully


## Load run_all_tools_cv

In [3]:
def run_all_tools_cv(adata: ad.AnnData, tools_to_run: List[str] = None, 
                    cv_config: Dict = None) -> Dict[str, Dict]:
    """
    Main Benchmarking Execution Function
    
    Purpose: Orchestrates the complete benchmarking workflow
    Inputs:
      - adata: AnnData object with Ground_Truth_Celltype in obs
      - tools_to_run: List of tool names to benchmark (default: TOOLS_TO_RUN)
      - cv_config: Dict with k, group_by, stratify_by, seed (default: CV_CONFIG)
    Outputs: Dict of results from each tool (pass to aggregate_all_tools())
    Workflow:
      1. Creates k-fold splits once (reused for all tools)
      2. Runs each tool across all folds with error handling
      3. Returns aggregated results ready for comparison
    """
    
    if tools_to_run is None:
        tools_to_run = TOOLS_TO_RUN
    if cv_config is None:
        cv_config = CV_CONFIG
    
    print("=== CELL TYPE ANNOTATION BENCHMARKING ===")
    print(f"Dataset: {len(adata.obs)} cells x {len(adata.var)} genes")
    
    if 'Ground_Truth_Celltype' in adata.obs.columns:
        unique_types = adata.obs['Ground_Truth_Celltype'].unique()
        print(f"Cell types: {', '.join(map(str, unique_types))}")
    
    print(f"Tools to run: {', '.join(tools_to_run)}")
    
    # Create folds once, reuse for all tools
    folds = create_cv_folds(adata, 
                           k=cv_config['k'],
                           group_by=cv_config['group_by'], 
                           stratify_by=cv_config['stratify_by'],
                           seed=cv_config['seed'])
    
    # Initialize results storage
    all_tool_results = {}
    
    # Run each tool
    for tool_name in tools_to_run:
        tool_function = TOOL_REGISTRY.get(tool_name)
        if tool_function is None:
            warnings.warn(f"Tool {tool_name} not found in registry")
            continue
        
        # Run CV for this tool
        tool_results = run_single_tool_cv(adata, tool_function, folds, tool_name)
        all_tool_results[tool_name] = tool_results
    
    return all_tool_results

## Single 80/20 Split Execution (L9 Simulated Datasets)

In [4]:
def run_all_tools_single_split(
    adata: ad.AnnData,
    tools_to_run: List[str] = None,
    label_col: str = "Ground_Truth_Celltype",
    seed: int = 123
) -> Dict[str, Dict]:
    """
    Single 80/20 Split Execution — for L9 simulated datasets.

    Purpose: Runs all tools on one stratified 80/20 train/test split.
             Used for L9 benchmark replicates where each dataset is an
             independent Splatter simulation — CV is not needed since
             fold variance is near-zero for these dataset sizes, and
             independent replicates capture between-simulation variance
             more informatively.
    Inputs:
      - adata: AnnData with label_col in obs
      - tools_to_run: list of tool names from TOOL_REGISTRY
      - label_col: obs column containing ground truth labels
      - seed: random seed for reproducibility
    Outputs: Dict in the same structure as run_all_tools_cv() — compatible
             with aggregate_all_tools() without modification.
    """

    if tools_to_run is None:
        tools_to_run = TOOLS_TO_RUN

    print("=== CELL TYPE ANNOTATION BENCHMARKING (Single 80/20 Split) ===")
    print(f"Dataset: {len(adata.obs)} cells x {len(adata.var)} genes")

    if label_col in adata.obs.columns:
        unique_types = adata.obs[label_col].unique()
        print(f"Cell types: {', '.join(map(str, unique_types))}")

    print(f"Tools to run: {', '.join(tools_to_run)}")

    # Stratified 80/20 split using StratifiedKFold(n_splits=5)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    all_indices = np.arange(len(adata.obs))
    stratify_labels = adata.obs[label_col]
    train_idx, test_idx = next(skf.split(all_indices, stratify_labels))

    adata_train = adata[train_idx].copy()
    adata_test = adata[test_idx].copy()
    print(f"Train: {adata_train.n_obs} cells, Test: {adata_test.n_obs} cells")

    # Find marker genes on training data
    try:
        sc.tl.rank_genes_groups(adata_train, groupby=label_col,
                               method='wilcoxon', use_raw=False)
        markers = sc.get.rank_genes_groups_df(adata_train, group=None)
        print(f"Found {len(markers)} marker genes")
    except Exception as e:
        warnings.warn(f"Marker detection failed: {e}")
        markers = pd.DataFrame()

    # Run each tool
    all_tool_results = {}

    for tool_name in tools_to_run:
        tool_function = TOOL_REGISTRY.get(tool_name)
        if tool_function is None:
            warnings.warn(f"Tool {tool_name} not found in registry")
            continue

        print(f"\n=== Running {tool_name} (single split) ===")

        try:
            t0 = time.perf_counter()
            result = tool_function(adata_train, adata_test, markers)
            wall_time = time.perf_counter() - t0

            runtime = result.get('method_walltime', wall_time)
            peak_mem = result.get('peak_memory_mb', None)
            peak_vram = result.get('peak_vram_mb', None)

            predictions = result['predictions']
            true_labels = result['true_labels']
            confidence_scores = result.get('confidence_scores', [0.0] * len(predictions))
            cell_ids = result.get('cell_ids', list(adata_test.obs.index))

            metrics = calculate_metrics(
                predicted=predictions,
                true_labels=true_labels
            )

            # Build result structure matching aggregate_tool_results() output
            pooled_metrics = dict(metrics)
            pooled_metrics['confusion_matrix'] = None

            fold_variation = {
                'accuracy_mean': metrics['overall_accuracy'],
                'accuracy_std': np.nan,
                'accuracy_folds': [metrics['overall_accuracy']],
                'accuracy_ci': [np.nan, np.nan],
                'f1_mean': metrics['macro_f1'],
                'f1_std': np.nan,
                'f1_folds': [metrics['macro_f1']],
                'f1_ci': [np.nan, np.nan],
                'kappa_mean': metrics['cohens_kappa'],
                'kappa_std': np.nan,
                'kappa_folds': [metrics['cohens_kappa']],
                'kappa_ci': [np.nan, np.nan],
                'mcc_mean': metrics['mcc'],
                'mcc_std': np.nan,
                'mcc_folds': [metrics['mcc']],
                'rare_f1_mean': metrics['rare_type_f1'],
                'rare_f1_std': np.nan,
                'rare_f1_folds': [metrics['rare_type_f1']],
                'unassigned_mean': metrics['unassigned_rate'],
                'unassigned_std': np.nan,
                'unassigned_folds': [metrics['unassigned_rate']]
            }

            valid_mem = [peak_mem] if peak_mem is not None else []
            valid_vram = [peak_vram] if peak_vram is not None and peak_vram > 0 else []

            tool_result = {
                'tool_name': tool_name,
                'pooled_metrics': pooled_metrics,
                'fold_variation': fold_variation,
                'runtime_stats': {
                    'mean_seconds': runtime,
                    'std_seconds': np.nan,
                    'total_seconds': runtime,
                    'fold_runtimes': [runtime]
                },
                'memory_stats': {
                    'mean_mb': peak_mem if peak_mem is not None else np.nan,
                    'std_mb': np.nan,
                    'max_mb': peak_mem if peak_mem is not None else np.nan,
                    'min_mb': peak_mem if peak_mem is not None else np.nan,
                    'fold_memories': valid_mem
                },
                'vram_stats': {
                    'mean_mb': peak_vram if peak_vram is not None and peak_vram > 0 else np.nan,
                    'std_mb': np.nan,
                    'max_mb': peak_vram if peak_vram is not None and peak_vram > 0 else np.nan,
                    'fold_vrams': valid_vram
                },
                'detailed_results': {
                    'predictions': predictions,
                    'true_labels': true_labels,
                    'confidence_scores': confidence_scores,
                    'cell_ids': cell_ids
                },
                'num_successful_folds': 1,
                'num_total_folds': 1
            }

            all_tool_results[tool_name] = tool_result

            memory_str = f", RAM: {peak_mem:.0f} MB" if peak_mem is not None else ""
            vram_str = f", VRAM: {peak_vram:.0f} MB" if peak_vram else ""
            print(f"    Completed in {runtime:.2f}s, accuracy: {metrics['overall_accuracy']*100:.2f}%{memory_str}{vram_str}")

        except Exception as e:
            warnings.warn(f"Tool {tool_name} failed: {str(e)}")
            print(f"    FAILED: {str(e)}")

            # Store all-NaN result
            nan_variation = {
                'accuracy_mean': np.nan, 'accuracy_std': np.nan, 'accuracy_folds': [],
                'accuracy_ci': [np.nan, np.nan],
                'f1_mean': np.nan, 'f1_std': np.nan, 'f1_folds': [],
                'f1_ci': [np.nan, np.nan],
                'kappa_mean': np.nan, 'kappa_std': np.nan, 'kappa_folds': [],
                'kappa_ci': [np.nan, np.nan],
                'mcc_mean': np.nan, 'mcc_std': np.nan, 'mcc_folds': [],
                'rare_f1_mean': np.nan, 'rare_f1_std': np.nan, 'rare_f1_folds': [],
                'unassigned_mean': np.nan, 'unassigned_std': np.nan, 'unassigned_folds': []
            }
            all_tool_results[tool_name] = {
                'tool_name': tool_name,
                'pooled_metrics': {
                    'overall_accuracy': np.nan, 'macro_f1': np.nan,
                    'cohens_kappa': np.nan, 'mcc': np.nan,
                    'unassigned_rate': np.nan, 'rare_type_f1': np.nan,
                    'rare_type_name': None,
                    'per_class_metrics': pd.DataFrame(),
                    'confusion_matrix': None
                },
                'fold_variation': nan_variation,
                'runtime_stats': {
                    'mean_seconds': np.nan, 'std_seconds': np.nan,
                    'total_seconds': np.nan, 'fold_runtimes': []
                },
                'memory_stats': {
                    'mean_mb': np.nan, 'std_mb': np.nan,
                    'max_mb': np.nan, 'min_mb': np.nan, 'fold_memories': []
                },
                'vram_stats': {
                    'mean_mb': np.nan, 'std_mb': np.nan,
                    'max_mb': np.nan, 'fold_vrams': []
                },
                'detailed_results': {
                    'predictions': [], 'true_labels': [],
                    'confidence_scores': [], 'cell_ids': []
                },
                'num_successful_folds': 0,
                'num_total_folds': 1
            }

    return all_tool_results

## Tool Registry and Execution

In [5]:
# Tool registry - populate with converted tool functions
TOOL_REGISTRY = {}

# Add Python DL-based tool functions (first batch)
if run_scDeepSort_function is not None:
    TOOL_REGISTRY["scDeepSort"] = run_scDeepSort_function

if run_scHash_function is not None:
    TOOL_REGISTRY["scHash"] = run_scHash_function

if run_scDeepinsight_function is not None:
    TOOL_REGISTRY["scDeepinsight"] = run_scDeepinsight_function

# Add Python DL-based tool functions (second batch)
if run_TOSICA_function is not None:
    TOOL_REGISTRY["TOSICA"] = run_TOSICA_function

if run_scMMT_function is not None:
    TOOL_REGISTRY["scMMT"] = run_scMMT_function

if run_scnym_function is not None:
    TOOL_REGISTRY["scnym"] = run_scnym_function

if run_CIForm_function is not None:
    TOOL_REGISTRY["CIForm"] = run_CIForm_function

if run_mtANN_function is not None:
    TOOL_REGISTRY["mtANN"] = run_mtANN_function

if run_scTransSort_function is not None:
    TOOL_REGISTRY["scTransSort"] = run_scTransSort_function

if run_scBalance_function is not None:
    TOOL_REGISTRY["scBalance"] = run_scBalance_function

if run_MARS_function is not None:
    TOOL_REGISTRY["MARS"] = run_MARS_function

if run_Cell_BLAST_function is not None:
    TOOL_REGISTRY["Cell_BLAST"] = run_Cell_BLAST_function

if run_SCANVI_function is not None:
    TOOL_REGISTRY["SCANVI"] = run_SCANVI_function

if run_ItClust_function is not None:
    TOOL_REGISTRY["ItClust"] = run_ItClust_function

if run_ACTINN_function is not None:
    TOOL_REGISTRY["ACTINN"] = run_ACTINN_function

# Add other tool functions as they are implemented
# TOOL_REGISTRY["CellTypist"] = run_CellTypist_function

print(f"Tools registered: {list(TOOL_REGISTRY.keys())}")
print(f"Total DL tools available: {len(TOOL_REGISTRY)}")

Tools registered: ['scDeepSort', 'scHash', 'scDeepinsight', 'TOSICA', 'scMMT', 'scnym', 'CIForm', 'mtANN', 'scTransSort', 'scBalance', 'MARS', 'Cell_BLAST', 'SCANVI', 'ItClust', 'ACTINN']
Total DL tools available: 15


## Configuration Section

In [6]:
# Data paths - Datasets (HDF5 files with normalized data)
#ADATA_PATH = "misc/AhernCOVID_Raw/AhernCOVID.h5ad"
ADATA_PATH = "data/S1_rep2_anndata_subset.h5ad"
# Cross-validation configuration
CV_CONFIG = {
    'k': 3,                              # Number of folds
    'group_by': None,                    # For grouped CV, set to None for stratified
    'stratify_by': "Ground_Truth_Celltype", # For stratified CV
    'seed': 123                          # Reproducibility
}


In [7]:
# L9 simulation split configuration
# Used by run_all_tools_single_split() for Splatter-generated datasets.
# Each L9 replicate is an independent dataset — no CV needed.
# Between-replicate variance (3 replicates per scenario) captures
# simulation variance more informatively than within-dataset CV folds.
SPLIT_CONFIG = {
    'label_col':   'Ground_Truth_Celltype',
    'train_prop':  0.80,
    'seed':        123
}

## Main Benchmarking Execution Function

In [ ]:
# BASIC WORKFLOW:
#
# 1. Load data
adata = sc.read_h5ad(ADATA_PATH)
#
# 2. Run benchmarking (start with subset for testing)
TOOLS_TO_RUN = [
    'scDeepSort', 'scHash', 'TOSICA', 'scMMT', 'scnym', 'CIForm','scBalance', 'Cell_BLAST', 'SCANVI'
]
results = run_all_tools_single_split(adata, tools_to_run= TOOLS_TO_RUN)
#
# 3. Aggregate and compare
final_results = aggregate_all_tools(results)
#
# 4. View results
print(final_results['comparison_table'])
print(final_results['summary_stats'])
#
# 5. Save results
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
with open(f"benchmarking_results_{timestamp}.pkl", 'wb') as f:
    pickle.dump(final_results, f)
final_results['comparison_table'].to_csv(f"DL_based2_tool_comparison_{timestamp}.csv", index=False)


=== CELL TYPE ANNOTATION BENCHMARKING (Single 80/20 Split) ===
Dataset: 500 cells x 2000 genes
Cell types: Type_Group3, Type_Group5, Type_Group4, Type_Group1, Type_Group2
Tools to run: scDeepSort, scHash, TOSICA, scMMT, scnym, CIForm, scBalance, Cell_BLAST, SCANVI
Train: 400 cells, Test: 100 cells


/home/oliver/miniconda3/envs/Benchmarking_Main/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:482: RuntimeWarning: invalid value encountered in log2
  self.stats[group_name, "logfoldchanges"] = np.log2(
/home/oliver/miniconda3/envs/Benchmarking_Main/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:482: RuntimeWarning: invalid value encountered in log2
  self.stats[group_name, "logfoldchanges"] = np.log2(
/home/oliver/miniconda3/envs/Benchmarking_Main/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:482: RuntimeWarning: invalid value encountered in log2
  self.stats[group_name, "logfoldchanges"] = np.log2(
/home/oliver/miniconda3/envs/Benchmarking_Main/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:482: RuntimeWarning: invalid value encountered in log2
  self.stats[group_name, "logfoldchanges"] = np.log2(
/home/oliver/miniconda3/envs/Benchmarking_Main/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:482: Runt

Found 10000 marker genes

=== Running scDeepSort (single split) ===
scDeepSort completed successfully:
  - Total predictions: 100
  - Unknown predictions: 0
  - Unique predictions: 1
    Completed in 30.43s, accuracy: 0.00%, RAM: 664 MB

=== Running scHash (single split) ===
